# Gold Customer Dimension

This notebook builds the `dim_customers` Gold model from the cleaned Silver customers dataset.

**Grain:** One row per `customer_id`.

In [0]:
from pyspark.sql import functions as F

## 1. Define storage paths

In [0]:
SILVER_CUSTOMERS_PATH = (
    "abfss://silver@stnovacartdev.dfs.core.windows.net/"
    "olist/customers"
)

GOLD_DIM_CUSTOMERS_PATH = (
    "abfss://gold@stnovacartdev.dfs.core.windows.net/"
    "olist/dim_customers"
)

print(f"Silver source: {SILVER_CUSTOMERS_PATH}")
print(f"Gold target: {GOLD_DIM_CUSTOMERS_PATH}")

## 2. Read Silver customers

In [0]:
silver_customers_df = (
    spark.read
    .format("delta")
    .load(SILVER_CUSTOMERS_PATH)
)

silver_customer_count = silver_customers_df.count()

print(f"Silver customer rows: {silver_customer_count:,}")

display(silver_customers_df.limit(10))

## 3. Validate required columns

In [0]:
required_columns = {
    "customer_id",
    "customer_unique_id",
    "customer_zip_code_prefix",
    "customer_city",
    "customer_state",
    "_silver_processed_at",
}

missing_columns = required_columns - set(silver_customers_df.columns)

if missing_columns:
    raise ValueError(
        f"Silver customers is missing required columns: {sorted(missing_columns)}"
    )

print("Required column validation passed.")

## 4. Validate customer state values

In [0]:
customer_states_df = (
    silver_customers_df
    .select("customer_state")
    .distinct()
    .orderBy("customer_state")
)

display(customer_states_df)

customer_state_count = customer_states_df.count()

print(f"Distinct customer states: {customer_state_count}")

valid_brazil_states = [
    "AC", "AL", "AP", "AM", "BA", "CE", "DF", "ES", "GO",
    "MA", "MT", "MS", "MG", "PA", "PB", "PR", "PE", "PI",
    "RJ", "RN", "RS", "RO", "RR", "SC", "SP", "SE", "TO"
]

unexpected_states_df = (
    silver_customers_df
    .filter(
        F.col("customer_state").isNull()
        | ~F.col("customer_state").isin(valid_brazil_states)
    )
    .select("customer_state")
    .distinct()
)

unexpected_state_count = unexpected_states_df.count()

if unexpected_state_count > 0:
    display(unexpected_states_df)
    raise ValueError(
        f"Found {unexpected_state_count} unexpected or null customer state values."
    )

print("Customer state validation passed.")

## 5.Build customer dimension

In [0]:
dim_customers_df = (
    silver_customers_df
    .select(
        "customer_id",
        "customer_unique_id",
        "customer_zip_code_prefix",
        "customer_city",
        "customer_state",
        "_silver_processed_at",
    )
    .withColumn(
        "customer_region",
        F.when(F.col("customer_state").isin("AC", "AP", "AM", "PA", "RO", "RR", "TO"), "North")
        .when(F.col("customer_state").isin("AL", "BA", "CE", "MA", "PB", "PE", "PI", "RN", "SE"), "Northeast")
        .when(F.col("customer_state").isin("DF", "GO", "MT", "MS"), "Central-West")
        .when(F.col("customer_state").isin("ES", "MG", "RJ", "SP"), "Southeast")
        .when(F.col("customer_state").isin("PR", "RS", "SC"), "South")
        .otherwise("Unknown")
    )
    .withColumn("_gold_processed_at", F.current_timestamp())
)

display(dim_customers_df.limit(10))

## 6. Validate customer dimension

In [0]:
dim_customer_count = dim_customers_df.count()

duplicate_customer_count = (
    dim_customers_df
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

null_customer_id_count = (
    dim_customers_df
    .filter(F.col("customer_id").isNull())
    .count()
)

if dim_customer_count == 0:
    raise ValueError("Customer dimension is empty.")

if dim_customer_count != silver_customer_count:
    raise ValueError(
        "Customer dimension row count does not match Silver customers. "
        f"Silver: {silver_customer_count:,}, Gold: {dim_customer_count:,}"
    )

if duplicate_customer_count > 0:
    raise ValueError(
        f"Customer dimension contains {duplicate_customer_count:,} duplicate customer IDs."
    )

if null_customer_id_count > 0:
    raise ValueError(
        f"Customer dimension contains {null_customer_id_count:,} null customer IDs."
    )

print(f"Customer dimension rows: {dim_customer_count:,}")
print("Customer dimension grain validation passed.")

## 7. Write customer dimension to Gold

In [0]:
(
    dim_customers_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(GOLD_DIM_CUSTOMERS_PATH)
)

print(f"Customer dimension written to: {GOLD_DIM_CUSTOMERS_PATH}")

## 8. Validate Gold output

In [0]:
written_dim_customers_df = (
    spark.read
    .format("delta")
    .load(GOLD_DIM_CUSTOMERS_PATH)
)

written_customer_count = written_dim_customers_df.count()

if written_customer_count != dim_customer_count:
    raise ValueError(
        "Gold customer dimension write validation failed. "
        f"Expected: {dim_customer_count:,}, Written: {written_customer_count:,}"
    )

print(f"Written customer dimension rows: {written_customer_count:,}")
print("Gold customer dimension write validation passed.")

## 9. Inspect Gold customer dimension

In [0]:
written_dim_customers_df.printSchema()

display(
    written_dim_customers_df
    .orderBy("customer_id")
    .limit(10)
)